<h1 style='color:#A52A2A'>Generating Paligrams</h1> 
<h3 style='color:#A52A2A'>A companion to <small><a href="https://oeis.org/A400282">A400282</a></small>.</h3>
<h4 style='color:#708090'>Peter Luschny, October 2026</h4>

A palindromic diagram of n is an edge-connected, self-conjugate skew Young/Ferrers diagram with n cells, if the string formed by concatenating the rows is a palindromic word over a two-symbol alphabet. We suggest calling these objects 'paligrams', and we will use this new name here.

For example, the following palindromic diagram with 10 cells

```python
  o o * * 
  o * * * 
  * * * o 
  * * o o 
```
forms the palindromic string 'oo**o******o**oo' when read by rows. Reversing the concatenated string reverses the row order and each row.

Self-conjugacy means the diagram is invariant under reflection in the main diagonal (transposition), and palindromicity means it is invariant under the half-turn (rotation by 180 degrees). Together, these force a third symmetry, the reflection in the antidiagonal. Hence a paligram is invariant under the full Klein four-group (Vierergruppe) generated by transposition and the half-turn, acting on its bounding box. The bounding box, i.e., the smallest rectangle containing the diagram, is a square. Thus a paligram is symmetric with respect to both diagonals of this square and under the half-turn about its centre.

Each row is a single contiguous nonempty run of * cells (this follows from the skew shape and the edge-connectivity), and by symmetry the same holds for each column. The top row touches the right edge, and the bottom row touches the left edge. As one moves downward, both ends of the run move weakly left, and consecutive runs share a column.

Theorem: For each n >= 19 at least one paligram exists. (No paligrams exist for n in {0, 2, 3, 5, 6, 8, 11, 12, 18}.)

In [82]:
from math import isqrt

def hooks(h, m, i=1, cap=10**9):
    """Strictly decreasing a_1 > a_2 > ... >= 0, a_i <= h-1-2i, sum(2*a_i+1) = m."""
    if m == 0: yield ()
    for a in range(min(cap, h - 1 - 2 * i, (m - 1) // 2), isqrt(max(m - 1, 0)) - 1, -1):
        for t in hooks(h, m - 2 * a - 1, i + 1, a - 1):
            yield (a,) + t

def paligrams(n: int, cell="●", exterior="○"):
    """Yield (widths, rows) of all palindromic diagrams with n cells."""
    for h in range(1, (n + 2) // 3 + 1):  # n >= 3h-2
        m = h * h - n                     # n = h^2 - 2|mu|
        if m < 0 or m % 2: continue
        found = []; A = [()]
        for A in hooks(h, m // 2):        # mu = self-conjugate, given by its hooks
            S = {(i, j) for i, a in enumerate(A, 1) for j in range(1, a + i + 1)}
            S |= {(j, i) for i, j in S}   # add the columns (transpose)
            mu = [sum((r, c) in S for c in range(1, h + 1)) for r in range(1, h + 1)]
            found.append(tuple((mu[r] + 1, h - mu[h - 1 - r]) for r in range(h)))
        for rows in sorted(found):
            yield ([R - L + 1 for L, R in rows],
                   [''.join(cell if L <= c <= R else exterior for c in range(1, h + 1)) for L, R in rows],
                   A)

In [83]:
def show_paligrams(n: int, flat: bool=False, info: bool=False):
    p = list(paligrams(n))
    if not p: return
    if flat: 
        for l in p:
            print(f"{n:4} {l[1]}")
        return
    for l in p:
        if not l: continue
        if info:
            print(n, len(l[0]), l[0], l[2])
        else:
            print(n, len(l[0]))
        for s in l[1]:
            print("  ", s)

In [84]:
# Example: show paligrams as squares.
show_paligrams(27)

27 7
   ○○○○●●●
   ○○○●●●●
   ○○○●●●●
   ○●●●●●○
   ●●●●○○○
   ●●●●○○○
   ●●●○○○○
27 9
   ○○○○○○○●●
   ○○○○○○●●●
   ○○○○○●●●○
   ○○○●●●●○○
   ○○○●●●○○○
   ○○●●●●○○○
   ○●●●○○○○○
   ●●●○○○○○○
   ●●○○○○○○○


In [85]:
# Example: show paligrams as squares plus info about structure.
show_paligrams(27, info=True)

27 7 [3, 4, 4, 5, 4, 4, 3] (3, 1, 0)
   ○○○○●●●
   ○○○●●●●
   ○○○●●●●
   ○●●●●●○
   ●●●●○○○
   ●●●●○○○
   ●●●○○○○
27 9 [2, 3, 3, 4, 3, 4, 3, 3, 2] (6, 4, 2)
   ○○○○○○○●●
   ○○○○○○●●●
   ○○○○○●●●○
   ○○○●●●●○○
   ○○○●●●○○○
   ○○●●●●○○○
   ○●●●○○○○○
   ●●●○○○○○○
   ●●○○○○○○○


In [86]:
# Examples: show paligrams as lists of rows.
for n in range(30):
    show_paligrams(n, flat=True)

   1 ['●']
   4 ['●●', '●●']
   7 ['○●●', '●●●', '●●○']
   9 ['●●●', '●●●', '●●●']
  10 ['○○●●', '○●●●', '●●●○', '●●○○']
  13 ['○○○●●', '○○●●●', '○●●●○', '●●●○○', '●●○○○']
  14 ['○●●●', '●●●●', '●●●●', '●●●○']
  15 ['○○○●●', '○●●●●', '○●●●○', '●●●●○', '●●○○○']
  16 ['●●●●', '●●●●', '●●●●', '●●●●']
  16 ['○○○○●●', '○○○●●●', '○○●●●○', '○●●●○○', '●●●○○○', '●●○○○○']
  17 ['○○●●●', '○○●●●', '●●●●●', '●●●○○', '●●●○○']
  19 ['○○●●●', '○●●●●', '●●●●●', '●●●●○', '●●●○○']
  19 ['○○○○○●●', '○○○○●●●', '○○○●●●○', '○○●●●○○', '○●●●○○○', '●●●○○○○', '●●○○○○○']
  20 ['○○○●●●', '○○○●●●', '○○●●●●', '●●●●○○', '●●●○○○', '●●●○○○']
  20 ['○○○○●●', '○○●●●●', '○●●●●○', '○●●●●○', '●●●●○○', '●●○○○○']
  21 ['○○○○○●●', '○○○○●●●', '○○●●●●○', '○○●●●○○', '○●●●●○○', '●●●○○○○', '●●○○○○○']
  22 ['○○○○●●', '○●●●●●', '○●●●●○', '○●●●●○', '●●●●●○', '●●○○○○']
  22 ['○○○○○○●●', '○○○○○●●●', '○○○○●●●○', '○○○●●●○○', '○○●●●○○○', '○●●●○○○○', '●●●○○○○○', '●●○○○○○○']
  23 ['○●●●●', '●●●●●', '●●●●●', '●●●●●', '●●●●○']
  23 ['○○○○●●●',